# CLIP Playground (pretrained OpenAI CLIP), beginner annotated version

Every code line has a comment (`# ...`) explaining what it does. Read the markdown cell
above each code cell first, then run the cell (Shift+Enter).

**Runtime:** `Runtime > Change runtime type > T4 GPU` (CPU works but is slow).

## 0. The big idea in 2 minutes

**What is CLIP?** A model from OpenAI trained on 400 million (image, caption) pairs from the internet.
It has two parts:

- an **image encoder** that turns a picture into a list of 512 numbers (a *vector*, also called an *embedding*)
- a **text encoder** that turns a sentence into a list of 512 numbers too

Training pushed the two encoders so that a picture and a caption that *describe the same thing* end
up with **similar vectors**, and unrelated ones end up with different vectors.

**Why that's useful:** to classify a photo, you don't need to train anything. Write the class names
as sentences ("a photo of a cat", "a photo of a dog"), encode them, encode the image, and pick the
sentence whose vector is closest to the image vector. That is called **zero-shot classification**
("zero" labelled training examples).

**How do we measure "closest"?** With **cosine similarity**: we first rescale each vector to length 1
(*normalising*), then take the **dot product** (multiply matching numbers and add them up).
The result is between -1 and 1: higher means more similar.

**Words you will see:**

| Term | Meaning |
|---|---|
| tensor | PyTorch's version of an array (like a NumPy array that can live on the GPU) |
| embedding / features | the vector of numbers a model produces for an input |
| `[N, D]` | shape of a tensor: N rows (one per item), D columns (numbers per item) |
| batch | a group of inputs processed together, for speed |
| logits | raw scores before converting to probabilities |
| softmax | turns a list of scores into probabilities that add up to 1 |
| accuracy | fraction of predictions that are correct |
| few-shot | learning from only a few labelled examples per class (k-shot = k examples) |
| linear probe | keep CLIP frozen, train a tiny simple classifier on top of its embeddings |

## 1. Setup

Lines starting with `!` are **shell commands**, not Python. `pip install` downloads libraries.
`-q` means "quiet" (less output).

- `transformers`: Hugging Face's library that gives us pretrained CLIP
- `scikit-learn`: classic ML library; we use its logistic regression and metrics

In [ ]:
!pip -q install transformers scikit-learn

Now we **import** the libraries (make them available in this notebook) and check for a GPU.

In [ ]:
import os, time                       # os: work with files/folders; time: measure how long things take
from io import BytesIO                # lets us treat downloaded bytes like a file

import numpy as np                    # NumPy: fast arrays and math on the CPU ("np" is the usual nickname)
import requests                       # downloads things from the internet (images from URLs)
import torch                          # PyTorch: the deep learning library CLIP runs on
import matplotlib.pyplot as plt       # plotting library for images and charts
from PIL import Image                 # PIL (Pillow): opens and manipulates images
from transformers import CLIPModel, CLIPProcessor   # the CLIP model and its input preprocessor

# Use the GPU ("cuda") if Colab gave us one, otherwise fall back to the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)              # should print "cuda" on a GPU runtime
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))   # name of the GPU, e.g. "Tesla T4"

## 2. Load the model

`from_pretrained` downloads the **trained weights** (the millions of numbers CLIP learned) the first
time, then loads them. Nothing is trained in this notebook's sections 2 to 5; we only *use* CLIP.

| MODEL_ID | Notes |
|---|---|
| `openai/clip-vit-base-patch32` | Fastest, ~150M params |
| `openai/clip-vit-base-patch16` | Better accuracy, slower |
| `openai/clip-vit-large-patch14` | Best of the three, needs GPU |

"ViT" = Vision Transformer (the image encoder's architecture). "patch32" means the image is cut into
32x32-pixel squares before processing; smaller patches = more detail = slower.

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"    # which pretrained checkpoint to use (change to try others)

# Download + load the model, move it to the GPU, and switch to evaluation mode
# (.eval() turns off training-only behaviour so results are deterministic)
model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()

# The processor prepares inputs: resizes/crops/normalises images to 224x224,
# and splits text into tokens (word pieces) that the text encoder understands
processor = CLIPProcessor.from_pretrained(MODEL_ID)

# Count parameters (learned numbers) in the model; 1e6 = one million
print(f"Loaded {MODEL_ID}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
# projection_dim = length of each embedding vector (512 for base models)
# logit_scale = a learned "temperature" (about 100) that sharpens probabilities, used later
print("Embedding dim:", model.config.projection_dim, "| logit scale:", model.logit_scale.exp().item())

### Helper functions

A **function** (`def name(...):`) is reusable code. We define four helpers here and use them
everywhere below. Nothing visible happens when you run this cell; it just defines them.

`@torch.no_grad()` is a **decorator** that tells PyTorch "we are not training, don't track gradients".
That saves memory and time.

In [ ]:
def _as_tensor(out, key):
    """Different versions of transformers return slightly different things.
    This makes sure we always get back a plain tensor of embeddings."""
    if torch.is_tensor(out):                  # older versions: already a tensor, return it
        return out
    for k in (key, "pooler_output"):          # newer versions: an object holding the tensor
        v = getattr(out, k, None)             # try to read the attribute named k
        if v is not None:
            return v
    raise TypeError(f"Unexpected output type: {type(out)}")   # give a clear error if neither worked

@torch.no_grad()
def encode_images(images, batch_size=64):
    """List of PIL images -> normalised embeddings, shape [N, D] (N images, D=512 numbers each)."""
    feats = []                                            # collect the results of each batch here
    for i in range(0, len(images), batch_size):           # i = 0, 64, 128, ... (step through in batches)
        batch = images[i:i + batch_size]                  # slice out up to 64 images
        # processor: resize + normalise pixels -> tensors; .to(device) moves them to the GPU
        inputs = processor(images=batch, return_tensors="pt").to(device)
        f = _as_tensor(model.get_image_features(**inputs), "image_embeds")   # run the image encoder
        # f.norm(...) = length of each vector; dividing makes every vector length 1 (normalising),
        # so a later dot product equals cosine similarity
        feats.append(f / f.norm(dim=-1, keepdim=True))
    return torch.cat(feats)                               # stack all batches into one [N, D] tensor

@torch.no_grad()
def encode_texts(texts, batch_size=256):
    """List of strings -> normalised embeddings, shape [N, D]. Same idea as encode_images."""
    feats = []
    for i in range(0, len(texts), batch_size):
        # padding=True: make all sentences in the batch the same length
        # truncation=True: cut sentences longer than CLIP's 77-token limit
        inputs = processor(text=texts[i:i + batch_size], return_tensors="pt",
                           padding=True, truncation=True).to(device)
        f = _as_tensor(model.get_text_features(**inputs), "text_embeds")    # run the text encoder
        feats.append(f / f.norm(dim=-1, keepdim=True))    # normalise to length 1
    return torch.cat(feats)

def load_image(src):
    """Open an image from a web URL or a file path, as RGB (3 colour channels)."""
    if src.startswith("http"):                            # it's a URL
        r = requests.get(src, timeout=30)                 # download it (give up after 30 s)
        r.raise_for_status()                              # raise an error if the download failed (e.g. 404)
        return Image.open(BytesIO(r.content)).convert("RGB")   # bytes -> image; force RGB
    return Image.open(src).convert("RGB")                 # it's a local file

## 3. Zero-shot classification

Steps:
1. For each class name, fill it into several sentence **templates** ("a photo of a {}.", "a blurry photo of a {}.", ...)
2. Encode all those sentences and **average** them into one vector per class (*prompt ensembling*; more robust than one sentence)
3. Encode the image
4. Similarity between the image vector and each class vector -> softmax -> probabilities

In [ ]:
# "{}" is a placeholder; .format("cat") turns "a photo of a {}." into "a photo of a cat."
TEMPLATES = [
    "a photo of a {}.",
    "a blurry photo of a {}.",
    "a close-up photo of a {}.",
    "a photo of the large {}.",
    "a photo of the small {}.",
    "a low resolution photo of a {}.",
    "a cropped photo of a {}.",
    "a bright photo of a {}.",
]

@torch.no_grad()
def build_zeroshot_weights(classnames, templates=TEMPLATES):
    """One averaged, normalised text vector per class. Output shape [C, D] (C = number of classes).
    These vectors act like the 'weights' of a classifier, which is why it's named this way."""
    weights = []
    for c in classnames:                                       # loop over each class name
        t = encode_texts([tpl.format(c) for tpl in templates]) # encode all 8 sentences -> [8, D]
        w = t.mean(0)                                          # average over the 8 rows -> [D]
        weights.append(w / w.norm())                           # re-normalise to length 1
    return torch.stack(weights)                                # list of C vectors -> [C, D] tensor

@torch.no_grad()
def zero_shot(image, classnames, templates=TEMPLATES, show=True, topk=5):
    W = build_zeroshot_weights(classnames, templates)          # class vectors, [C, D]
    img = encode_images([image])                               # image vector, [1, D]
    # img @ W.T  : "@" is matrix multiplication; W.T is W transposed to [D, C].
    #              Result [1, C] = cosine similarity of the image with every class.
    # * logit_scale : multiply by ~100 so differences become large enough for softmax to be confident
    # .softmax(-1)  : turn scores into probabilities summing to 1
    # [0]           : take the single row; .cpu().numpy() moves it to the CPU as a NumPy array
    probs = (model.logit_scale.exp() * img @ W.T).softmax(-1)[0].cpu().numpy()
    order = probs.argsort()[::-1][:topk]                       # indices sorted high->low, keep top k
    if show:
        # 1 row, 2 plots side by side: the image and a horizontal bar chart
        fig, ax = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1, 1.2]})
        ax[0].imshow(image); ax[0].axis("off")                 # show image without axes
        # [::-1] reverses so the highest bar ends up at the top
        ax[1].barh([classnames[i] for i in order][::-1], probs[order][::-1], color="#4c72b0")
        ax[1].set_xlim(0, 1); ax[1].set_xlabel("probability")
        ax[1].set_title("Zero-shot prediction")
        plt.tight_layout(); plt.show()
    # return a dictionary like {"cat": 0.93, "couch": 0.04, ...}
    return {classnames[i]: float(probs[i]) for i in order}

In [ ]:
# A COCO dataset photo of two cats on a couch with TV remotes
img = load_image("http://images.cocodataset.org/val2017/000000039769.jpg")
labels = ["cat", "dog", "remote control", "couch", "car", "bird", "pizza"]   # the candidate classes
zero_shot(img, labels)    # CLIP has never been told these are the classes; it just compares vectors

### Try your own image

Edit `MY_URL` and `MY_LABELS`, or run the upload cell after it. Labels can be any text, for example
`"a torn seam on a shirt"` vs `"a clean seam on a shirt"`. Note that probabilities are always
*relative to the labels you give*: if none fit, CLIP still picks one.

In [ ]:
MY_URL = "http://images.cocodataset.org/val2017/000000000285.jpg"   # a photo of a bear
MY_LABELS = ["bear", "dog", "teddy bear", "wolf", "person"]
zero_shot(load_image(MY_URL), MY_LABELS)

In [ ]:
# Upload images from your computer (opens a file picker in Colab)
try:                                          # "try" lets us handle the error if not in Colab
    from google.colab import files            # Colab-only module
    uploaded = files.upload()                 # dictionary: {filename: file bytes}
    for name in uploaded:                     # the files are saved in the current folder
        print(name)
        zero_shot(load_image(name), MY_LABELS)
except ImportError:                           # happens when running outside Colab
    print("Not running in Colab: set a local path and call zero_shot(load_image(path), MY_LABELS).")

## 4. Image-text similarity matrix

Encode 4 images and 4 captions, compute every image-caption similarity, and show it as a coloured grid.
Each caption was written for one image, so the **diagonal** (top-left to bottom-right) should be brightest.

In [ ]:
# Dictionary: image URL -> the caption that describes it
pairs = {
    "http://images.cocodataset.org/val2017/000000039769.jpg": "two cats sleeping on a pink couch",
    "http://images.cocodataset.org/val2017/000000000285.jpg": "a large brown bear sitting in the grass",
    "http://images.cocodataset.org/val2017/000000000724.jpg": "a red stop sign on a street",
    "http://images.cocodataset.org/val2017/000000000785.jpg": "a woman skiing down a snowy slope",
}
imgs, caps = [], []
for url, cap in pairs.items():                # loop over (url, caption) pairs
    try:
        imgs.append(load_image(url)); caps.append(cap)
    except Exception as e:                    # if one download fails, skip it instead of crashing
        print("skip", url, e)

# [4, D] @ [D, 4] -> [4, 4] grid. sim[i, j] = similarity(image i, caption j)
sim = (encode_images(imgs) @ encode_texts(caps).T).cpu().numpy()
# Note: rows are drawn as captions and columns as images below; since it's a 4x4 matched set,
# the diagonal is still "correct pair".

fig, ax = plt.subplots(figsize=(2.2 * len(imgs) + 4, 2.2 * len(imgs)))
ax.imshow(sim, cmap="viridis")                            # colour each cell by its value
ax.set_yticks(range(len(caps))); ax.set_yticklabels(caps) # captions as row labels
ax.set_xticks([])                                         # no column numbers; thumbnails instead
for j, im in enumerate(imgs):                             # draw a small thumbnail under each column
    ax.imshow(im.resize((100, 100)), extent=(j - 0.45, j + 0.45, len(caps) - 0.05, len(caps) + 0.85))
for i in range(sim.shape[0]):                             # write the number inside every cell
    for j in range(sim.shape[1]):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", color="white", fontsize=11)
ax.set_xlim(-0.5, len(imgs) - 0.5); ax.set_ylim(len(caps) + 0.9, -0.5)   # make room for thumbnails
ax.set_title("Cosine similarity (images x captions)")
plt.tight_layout(); plt.show()

## 5. Zero-shot benchmark on CIFAR-10

**CIFAR-10** is a classic dataset: 60,000 tiny (32x32) photos in 10 classes (airplane, car, bird, cat,
deer, dog, frog, horse, ship, truck). 50,000 are for training and 10,000 for testing.

Here we measure **accuracy**: how many of 2,000 random test images CLIP labels correctly, without training.
We encode the images once and keep the vectors (`test_feats`) because sections 6 and 7 reuse them.
Expect roughly 85 to 90% with ViT-B/32.

In [ ]:
from torchvision.datasets import CIFAR10       # torchvision has built-in downloaders for common datasets

N_TEST = 2000          # how many test images to use (raise to 10000 for the full set; slower)
SEED = 0               # fixed random seed so you get the same random sample every run
rng = np.random.default_rng(SEED)              # random number generator

# download=True fetches the data (~170 MB) the first time into /content/data
cifar_test = CIFAR10(root="/content/data", train=False, download=True)
cifar_train = CIFAR10(root="/content/data", train=True, download=True)
CLASSES = cifar_test.classes                   # ['airplane', 'automobile', 'bird', ...]

# pick 2000 random indices from 0..9999, no repeats
test_idx = rng.choice(len(cifar_test), N_TEST, replace=False)
# cifar_test[i] returns (PIL image, label number). [0] = image, [1] = label
test_imgs = [cifar_test[i][0] for i in test_idx]
test_labels = np.array([cifar_test[i][1] for i in test_idx])   # true answers, numbers 0..9

t0 = time.time()                               # start a timer
test_feats = encode_images(test_imgs)          # [2000, 512] embeddings
print(f"Encoded {N_TEST} test images in {time.time() - t0:.1f}s")

In [ ]:
def zeroshot_accuracy(feats, labels, classnames, templates=TEMPLATES):
    W = build_zeroshot_weights(classnames, templates)      # [10, D] class vectors
    # [2000, D] @ [D, 10] -> [2000, 10] scores; argmax picks the best class for each image
    preds = (feats @ W.T).argmax(-1).cpu().numpy()
    # (preds == labels) is an array of True/False; .mean() = fraction True = accuracy
    return (preds == labels).mean(), preds

# Compare: one template only vs all 8 templates averaged
acc_single, _ = zeroshot_accuracy(test_feats, test_labels, CLASSES, ["a photo of a {}."])
acc_ens, zs_preds = zeroshot_accuracy(test_feats, test_labels, CLASSES)
print(f"Zero-shot, single prompt : {acc_single * 100:.2f}%")
print(f"Zero-shot, {len(TEMPLATES)} prompts    : {acc_ens * 100:.2f}%")

A **confusion matrix** shows which classes get mixed up. Row = true class, column = predicted class.
Values are fractions of each row, so a perfect model has 1.00 on the diagonal and 0 elsewhere.
Look for bright off-diagonal cells (often cat vs dog).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(test_labels, zs_preds, normalize="true")   # normalize="true": divide each row by its total
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", values_format=".2f",
                                                       xticks_rotation=45, colorbar=False)
ax.set_title("CIFAR-10 zero-shot confusion matrix")
plt.tight_layout(); plt.show()

## 6. Few-shot linear probe

Now we **do** train something, but tiny: CLIP stays frozen (unchanged). We take **k** labelled training
images per class (k = 1, 2, 4, 8, 16), turn them into CLIP vectors, and train a **logistic regression**
on those vectors. Logistic regression is a simple classifier that learns one weight per number in
the vector, per class.

Since a random choice of k images can be lucky or unlucky, we repeat each k three times (*trials*) and
report the **mean +/- standard deviation**.

Typical finding: 1-shot is *worse* than zero-shot (one example is noisy, while the text prompt already
carries a lot of knowledge), and the probe overtakes zero-shot after a few shots.

In [ ]:
from sklearn.linear_model import LogisticRegression

SHOTS = [1, 2, 4, 8, 16]      # k values to try
N_TRIALS = 3                  # repetitions per k with different random images

train_targets = np.array(cifar_train.targets)   # labels of all 50,000 training images
X_test = test_feats.cpu().numpy()               # scikit-learn needs NumPy arrays, not GPU tensors

def sample_support(k, seed):
    """Pick k random training images from each of the 10 classes. Returns their indices.
    ('support set' is few-shot jargon for the few labelled examples you learn from.)"""
    r = np.random.default_rng(seed)
    idx = np.concatenate([
        r.choice(np.where(train_targets == c)[0], k, replace=False)   # k indices whose label is c
        for c in range(len(CLASSES))                                  # for each class c = 0..9
    ])
    return idx                                                        # 10*k indices in total

results = {}                                    # will hold {k: (mean accuracy, std)}
for k in SHOTS:
    accs = []
    for t in range(N_TRIALS):
        idx = sample_support(k, seed=100 * k + t)                     # different seed per trial
        X_tr = encode_images([cifar_train[i][0] for i in idx]).cpu().numpy()   # [10k, D] features
        y_tr = train_targets[idx]                                     # their labels
        # C controls regularisation (smaller C = simpler model, less overfitting).
        # max_iter = how many optimisation steps the solver may take.
        clf = LogisticRegression(C=0.316, max_iter=2000)
        clf.fit(X_tr, y_tr)                                           # TRAIN on the few examples
        accs.append(clf.score(X_test, test_labels))                   # accuracy on the 2000 test images
    results[k] = (np.mean(accs), np.std(accs))
    print(f"{k:>2}-shot: {results[k][0] * 100:.2f}% +/- {results[k][1] * 100:.2f}")

In [ ]:
# Plot accuracy vs number of shots, with the zero-shot result as a dashed reference line
ks = list(results)
means = [results[k][0] * 100 for k in ks]
stds = [results[k][1] * 100 for k in ks]
plt.figure(figsize=(6, 4))
plt.errorbar(ks, means, yerr=stds, marker="o", capsize=4, label="linear probe")   # error bars = std
plt.axhline(acc_ens * 100, color="gray", ls="--", label="zero-shot (ensemble)")   # horizontal line
plt.xscale("log", base=2); plt.xticks(ks, ks)    # log scale so 1,2,4,8,16 are evenly spaced
plt.xlabel("shots per class"); plt.ylabel("accuracy (%)")
plt.title(f"CIFAR-10: few-shot probe vs zero-shot ({MODEL_ID.split('/')[-1]})")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 7. Text-to-image retrieval (search)

The reverse direction: encode a text query, compare it with all 2,000 image vectors, and show the
best matches. This is how "search your photos by description" features work.

In [ ]:
def search(query, k=8):
    q = encode_texts([query])                      # [1, D] query vector
    scores = (test_feats @ q.T).squeeze(1)         # [2000, D] @ [D, 1] -> [2000, 1] -> squeeze -> [2000]
    top = scores.topk(k)                           # the k highest scores and their indices
    fig, axes = plt.subplots(1, k, figsize=(2 * k, 2.4))   # k small plots in a row
    for ax, s, i in zip(axes, top.values.cpu(), top.indices.cpu()):
        ax.imshow(test_imgs[i]); ax.axis("off")
        ax.set_title(f"{CLASSES[test_labels[i]]}\n{s:.3f}", fontsize=9)   # true class + score
    fig.suptitle(f'Query: "{query}"'); plt.tight_layout(); plt.show()

search("a red sports car")
search("a ship on the open sea")
search("a horse running in a field")

## 8. Your own dataset

Put images in a class-per-folder layout:

```
my_dataset/
  class_a/  img1.jpg img2.jpg ...
  class_b/  ...
```

Either upload a zip (`my_dataset.zip`) to `/content` (Files panel on the left), or mount Google Drive.
Folder names become class names for zero-shot, so make them readable (e.g. `hole_in_fabric`, `stain`).
Underscores become spaces.

**Cross-validation:** with a small dataset we can't afford a big separate test set. 5-fold CV splits the
data into 5 parts, trains on 4 and tests on 1, rotates 5 times, and averages. Every image gets tested once.

In [ ]:
# Option A: mount Google Drive (uncomment both lines by removing the "# ")
# from google.colab import drive; drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/my_dataset"

# Option B: upload my_dataset.zip to /content and unzip it
# !unzip -q /content/my_dataset.zip -d /content

DATA_DIR = "/content/my_dataset"     # folder that contains one sub-folder per class

In [ ]:
from glob import glob                                   # glob finds files matching a pattern like "*.jpg"
from sklearn.model_selection import StratifiedKFold     # CV splitter that keeps class proportions balanced

EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")   # file types we treat as images

if not os.path.isdir(DATA_DIR):                         # folder missing: tell the user and stop here
    print(f"{DATA_DIR} not found. Set DATA_DIR above and re-run.")
else:
    # every sub-folder is a class; sorted() keeps the order stable
    class_dirs = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
    my_classes = [d.replace("_", " ") for d in class_dirs]          # "hole_in_fabric" -> "hole in fabric"
    paths, ys = [], []                                               # image paths and their class numbers
    for ci, d in enumerate(class_dirs):                              # ci = 0, 1, 2, ... class index
        for p in sorted(glob(os.path.join(DATA_DIR, d, "*"))):       # every file in that class folder
            if p.lower().endswith(EXTS):                             # keep only image files
                paths.append(p); ys.append(ci)
    ys = np.array(ys)
    # np.bincount counts how many images per class
    print(f"{len(paths)} images, {len(my_classes)} classes:", dict(zip(my_classes, np.bincount(ys))))

    my_feats = encode_images([load_image(p) for p in paths])         # [N, D] embeddings of your images

    # Zero-shot using folder names (edit TEMPLATES or my_classes for domain-specific prompts)
    zs_acc, _ = zeroshot_accuracy(my_feats, ys, my_classes)
    print(f"Zero-shot accuracy: {zs_acc * 100:.2f}%")

    # Linear probe with 5-fold cross-validation (needs >= 5 images per class)
    if np.bincount(ys).min() >= 5:
        X = my_feats.cpu().numpy()
        accs = []
        # .split gives 5 pairs of (train indices, test indices)
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X, ys):
            clf = LogisticRegression(C=0.316, max_iter=2000).fit(X[tr], ys[tr])   # train on 4/5
            accs.append(clf.score(X[te], ys[te]))                                 # test on 1/5
        print(f"Linear probe (5-fold CV): {np.mean(accs) * 100:.2f}% +/- {np.std(accs) * 100:.2f}")
    else:
        print("Need at least 5 images per class for the cross-validated linear probe.")

## 9. Save embeddings (optional)

Encoding images is the slow part. Saving the vectors lets you reload them later with `torch.load`
instead of re-encoding. (Colab deletes `/content` when the session ends, so download the file or
save to Drive if you want to keep it.)

In [ ]:
torch.save({"model": MODEL_ID,                         # which model made these vectors
            "cifar_test_feats": test_feats.cpu(),      # the [2000, D] embeddings
            "cifar_test_labels": torch.tensor(test_labels)},
           "/content/clip_features.pt")
print("Saved /content/clip_features.pt")

## Ideas to try next

- Change `MODEL_ID` to `openai/clip-vit-large-patch14` and re-run: compare zero-shot and few-shot curves.
- Write domain-specific templates (e.g. `"a macro photo of {} on fabric."`) and see if zero-shot accuracy on your dataset improves.
- Try `open_clip` checkpoints (LAION / DataComp / SigLIP) for stronger backbones: `pip install open_clip_torch`.
- Replace the linear probe with Tip-Adapter or CoOp-style prompt tuning for better few-shot results.